# End-to-end system identification: a user-defined model

This notebook takes a pharmacokinetic model all the way from a plain Python function to
confidence intervals on its parameters, using `gsua_csb`'s `UserFunctionModel`.

The point is not that the fit succeeds. It is that **a good fit tells you almost nothing about
whether your parameters are identifiable** — and that the toolbox will tell you the difference
if you ask it.

The model is a one-compartment pharmacokinetic model with first-order absorption, the standard
description of an orally administered drug:

$$c(t)=\frac{D\,k_a}{V(k_a-k_e)}\left(e^{-k_e t}-e^{-k_a t}\right)$$

with three factors to identify — $k_a$ (absorption rate, 1/h), $k_e$ (elimination rate, 1/h)
and $V$ (apparent volume of distribution, L). The dose $D=100$ mg is known.

*A MATLAB Live Script version of this example is available alongside it.*

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from gsua_csb import (UserFunctionModel, parameter_estimation,
                      identifiability_analysis, profile_likelihood)

DOSE = 100.0

def pk_absorption(params, t):
    """One-compartment PK model with first-order absorption.

    params = [ka, ke, V]. Returns a (1, len(t)) array: one observed output, the
    plasma concentration.
    """
    ka, ke, V = params
    return ((DOSE * ka) / (V * (ka - ke)) * (np.exp(-ke * t) - np.exp(-ka * t)))[None, :]

## 1. Defining the model

`UserFunctionModel` wraps any callable. `range` gives the bounds the samplers and estimators
work within, one row per factor.

The $k_a$ range is deliberately kept above the $k_e$ range. At $k_a=k_e$ the closed form is
singular, and swapping the two leaves $c(t)$ unchanged — the classic *flip-flop* ambiguity.
Excluding it keeps this example about experimental design rather than an algebraic accident.

In [ ]:
TRUTH = np.array([1.2, 0.25, 15.0])
XDATA = np.array([0.25, 0.5, 1, 1.5, 2, 3, 4, 6, 8, 10, 12, 16, 20, 24.0])

model = UserFunctionModel(
    func=pk_absorption,
    names=["ka", "ke", "V"],
    range=np.array([[0.6, 3.0], [0.05, 0.5], [5.0, 40.0]]),
    nominal=TRUTH,
    domain=XDATA,
    output_names=["concentration"],
)
print(model.names, "| free factors:", (~model.fixed).sum())

## 2. Synthetic data

Using synthetic data means the truth is known, so the confidence intervals can be *checked*
rather than merely reported. The measurements carry 8% proportional noise.

In [ ]:
rng = np.random.default_rng(0)
clean = pk_absorption(TRUTH, XDATA)
ydata = clean * (1 + 0.08 * rng.standard_normal(clean.shape))

tdense = np.linspace(0.05, 24, 300)
plt.figure(figsize=(7, 4))
plt.plot(tdense, pk_absorption(TRUTH, tdense)[0], lw=1.8, label="true model")
plt.plot(XDATA, ydata[0], "ko", ms=5, label="measurements")
plt.xlabel("time (h)"); plt.ylabel("concentration (mg/L)")
plt.title("Simulated single-dose concentration data")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 3. First attempt: estimate all three factors

`parameter_estimation` runs a multistart fit. `margin=0.1` selects the correlation-penalized
cost and records the margin on the result, so downstream functions can recover it.

In [ ]:
pe3 = parameter_estimation(model, XDATA, ydata, n=20,
                           solver="least_squares", margin=0.1, seed=0)
best3 = pe3.x[np.argmin(pe3.cost)]

import pandas as pd
pd.DataFrame({"true": TRUTH, "estimated": best3.round(4)}, index=model.names)

In [ ]:
print(f"cost: min {pe3.cost.min():.5g}   max {pe3.cost.max():.5g}")
plt.figure(figsize=(7, 4))
plt.plot(tdense, pk_absorption(best3, tdense)[0], lw=1.8, label="fitted model")
plt.plot(XDATA, ydata[0], "ko", ms=5, label="measurements")
plt.xlabel("time (h)"); plt.ylabel("concentration (mg/L)")
plt.title(f"Fit with all three factors free (cost = {pe3.cost.min():.4g})")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

The fit is excellent and every multistart run converged to the same cost — usually taken as a
sign of a healthy estimation problem.

## 4. Diagnosis: is that fit identifiable?

The correlation between the repeated estimates is the first warning sign. Values near $\pm 1$
mean the factors trade off against each other: many different combinations reproduce the same
curve.

In [ ]:
ia3 = identifiability_analysis(model, pe3.x, cost=pe3.cost, cost_rtol=0.1, seed=0)
pd.DataFrame(ia3.correlation.round(4), index=model.names, columns=model.names)

`profile_likelihood` turns that into something actionable. It profiles each factor — stepping
it away from the estimate while re-fitting the others — and reports the interval where the fit
remains statistically acceptable.

In [ ]:
pl3 = profile_likelihood(model, XDATA, ydata, alpha=0.95, margin=0.1)
pd.DataFrame({
    "CI_low":  pl3.range[:, 0].round(4),
    "CI_high": pl3.range[:, 1].round(4),
    "width":   (pl3.range[:, 1] - pl3.range[:, 0]).round(4),
    "prior_low":  model.range[:, 0],
    "prior_high": model.range[:, 1],
}, index=model.names)

This is the result worth stopping on. The correlation between $k_a$ and $k_e$ is essentially
$-1$: the three factors cannot be separated from a single oral concentration curve, no matter
how well the curve fits. That is a textbook pharmacokinetic result, not a failure of the
optimizer.

## 5. Remedy: fix what an independent experiment already knows

The usual resolution is to determine $V$ separately, from an intravenous study where it *is*
directly identifiable, and estimate only the absorption and elimination constants. A factor is
fixed by giving it a degenerate range.

In [ ]:
model_fixed = UserFunctionModel(
    func=pk_absorption,
    names=["ka", "ke", "V"],
    range=np.array([[0.6, 3.0], [0.05, 0.5], [15.0, 15.0]]),   # V pinned
    nominal=TRUTH,
    domain=XDATA,
    output_names=["concentration"],
)
free = np.where(~model_fixed.fixed)[0]

pe2 = parameter_estimation(model_fixed, XDATA, ydata, n=20,
                           solver="least_squares", margin=0.1, seed=0)
best2 = pe2.x[np.argmin(pe2.cost)]
ia2 = identifiability_analysis(model_fixed, pe2.x, cost=pe2.cost, cost_rtol=0.1, seed=0)
pl2 = profile_likelihood(model_fixed, XDATA, ydata, alpha=0.95, margin=0.1,
                         params=list(free))

pd.DataFrame({
    "true":      TRUTH[free],
    "estimated": best2[free].round(4),
    "CI_low":    pl2.range[free, 0].round(4),
    "CI_high":   pl2.range[free, 1].round(4),
    "width":     (pl2.range[free, 1] - pl2.range[free, 0]).round(4),
}, index=[model.names[i] for i in free])

In [ ]:
comparison = pd.DataFrame({
    "all three free": [pe3.cost.min(), ia3.correlation[0, 1],
                       pl3.range[0, 1] - pl3.range[0, 0],
                       pl3.range[1, 1] - pl3.range[1, 0]],
    "V fixed":        [pe2.cost.min(), ia2.correlation[0, 1],
                       pl2.range[0, 1] - pl2.range[0, 0],
                       pl2.range[1, 1] - pl2.range[1, 0]],
}, index=["best cost", "corr(ka, ke)", "CI width ka", "CI width ke"]).round(4)
comparison

In [ ]:
labels = ["ka", "ke"]
x = np.arange(2); w = 0.35
plt.figure(figsize=(6, 4))
plt.bar(x - w/2, [pl3.range[i, 1] - pl3.range[i, 0] for i in range(2)], w, label="all three free")
plt.bar(x + w/2, [pl2.range[i, 1] - pl2.range[i, 0] for i in range(2)], w, label="V fixed")
plt.xticks(x, labels); plt.ylabel("95% confidence interval width")
plt.title("Fixing one factor sharpens the other two")
plt.legend(); plt.grid(alpha=.3, axis="y"); plt.tight_layout(); plt.show()

## 6. What the example shows

Fixing $V$ made the fit very slightly **worse** and the science considerably **better**: the
correlation between $k_a$ and $k_e$ collapses from about $-1$ to a manageable value, both
intervals tighten, and the estimates move closer to the truth.

Cost measures how well a curve passes through points. It does not measure whether the factors
that produced that curve could have been recovered. Only the identifiability analysis answers
that, which is why it belongs *inside* the workflow rather than after it.

> **Note on the MATLAB companion.** The same problem run through MATLAB's `gsua_likelihood`
> reports $k_a$'s interval as spanning its entire prior range in the all-free case. The two
> profile-likelihood implementations use different thresholds and stepping strategies, so
> interval *widths* are not directly comparable between the languages. What transfers is the
> ordering and the conclusion: strongly correlated factors, intervals that tighten sharply once
> $V$ is fixed.

Where a multistart run does spread across parameter space rather than converging to a point,
`identifiability_analysis` adds correlation structure and detection of multiple global minima,
`noise_floor` calibrates which fits to accept against the observation noise, and
`design_matrix(..., method="joint")` propagates the accepted set without destroying its
correlation structure.